# NB_lab_test_trends

Builds `<catalog>.gold.lab_test_trends` per the Silver to Gold STTM: one row per test type (test_name) per calendar month. It aggregates `silver.lab_results`, LEFT JOINs the calendar on the first day of the month, and adds a month-over-month LAG trend on abnormal rate. Load type: FULL (CREATE OR REPLACE TABLE ... AS SELECT).

Note: the STTM names the calendar key `date_key`, but the column in `silver.calendar` is actually `datekey` (an int in yyyyMMdd format). The LAG follows the STTM: it uses the previous month that has data for that test_name, which may not be the month directly before.

In [0]:
dbutils.widgets.text("catalog", "caresync_databricks")
CATALOG = dbutils.widgets.get("catalog")

SILVER = f"{CATALOG}.silver"
TARGET_TABLE = f"{CATALOG}.gold.lab_test_trends"
print(f"Source: {SILVER}  ->  Target: {TARGET_TABLE}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {TARGET_TABLE}
COMMENT 'Gold: one row per test_name per month with volumes, abnormal/critical rates and month-over-month trend. Source: STTM Silver to Gold v1.0.'
AS
WITH monthly AS (
    SELECT
        test_name,
        year(test_date)                       AS year,
        month(test_date)                      AS month,
        COUNT(*)                              AS total_tests,
        COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
        COUNT_IF(result_status = 'Critical')  AS critical_test_count
    FROM {SILVER}.lab_results
    GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
    SELECT
        m.*,
        CAST(ROUND(m.abnormal_test_count * 100.0 / m.total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
        CAST(ROUND(m.critical_test_count * 100.0 / m.total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct
    FROM monthly m
),
with_lag AS (
    SELECT
        r.*,
        LAG(r.abnormal_rate_pct) OVER (PARTITION BY r.test_name ORDER BY r.year, r.month) AS prev_month_abnormal_rate_pct
    FROM rates r
)
SELECT
    w.test_name,
    CAST(w.year AS INT)                                                   AS year,
    CAST(w.month AS INT)                                                  AS month,
    c.month_name,
    c.quarter_name,
    CAST(w.total_tests AS INT)                                            AS total_tests,
    CAST(w.abnormal_test_count AS INT)                                    AS abnormal_test_count,
    CAST(w.critical_test_count AS INT)                                    AS critical_test_count,
    w.abnormal_rate_pct,
    w.critical_rate_pct,
    w.prev_month_abnormal_rate_pct,
    CASE
        WHEN w.prev_month_abnormal_rate_pct IS NULL THEN NULL
        ELSE CAST(w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct AS DECIMAL(6,2))
    END                                                                   AS abnormal_rate_change_pct
FROM with_lag w
LEFT JOIN {SILVER}.calendar c
       ON c.datekey = CAST(date_format(make_date(w.year, w.month, 1), 'yyyyMMdd') AS INT)
""")

print(f"Written {spark.table(TARGET_TABLE).count():,} rows to {TARGET_TABLE}")

In [0]:
import json

record_count = spark.table(TARGET_TABLE).count()

# Returned as the notebook output (visible to Jobs / ADF / dbutils.notebook.run callers)
# dbutils.notebook.exit(json.dumps({"table": TARGET_TABLE, "record_count": record_count}))
dbutils.notebook.exit(str(record_count))